In [29]:
import re
import numpy as np
import pandas as pd
import nibabel as nib
import matplotlib.pyplot as plt
from pathlib import Path
 
from nilearn.glm.first_level import make_first_level_design_matrix, FirstLevelModel
from nilearn.glm.contrasts import compute_fixed_effects
from nilearn.plotting import plot_stat_map, find_xyz_cut_coords
from nilearn.glm import threshold_stats_img

In [ ]:
# ==============================================================================
# SECTION 1 — TOP-LEVEL SETTINGS (only edit this section)
# ==============================================================================

# --- Choose your experiment ---
TASK = "joystick"   # "motif4limbs"  OR  "joystick"

# --- Choose your subject and runs to exclude ---
SUB          = "07" # "01", "02", "03", "04", "05", "06", "07", "08", "09", "10"                     
EXCLUDE_RUNS = []

# --- Space ---
# "MNI152NLin2009cAsym" for group/atlas work
# "T1w"                 for native space / clinical precision
SPACE = "MNI152NLin2009cAsym"

# --- Acquisition parameters (motif4limbs only) ---
TR       = 1.6   # seconds
BLANK_S  = 1.0
PAUSE_S  = 4.8
STIM_S   = 2.2

# --- GLM settings ---
SMOOTHING_FWHM = None   # No smoothing at 7T

# --- Statistical thresholds ---
# ┌─────────────────────────────────────────────────────────────────────────┐
# │  Recommended settings per method:                                       │
# │                                                                         │
# │  TASK          METHOD    ALPHA_FDR  ALPHA_BONF  CLUSTER_THRESHOLD       │
# │  motif4limbs   sequence  0.001      0.05        10                      │
# │  motif4limbs   behav.    0.001      0.05        10                      │
# │  joystick      events    0.05       0.2         10                      │
# │  joystick      motion    0.05       0.2         0   ← no cluster req.   │
# │                                                                         │
# │  WHY cluster=0 for motion?                                              │
# │  Parametric effects (amplitude encoding) are spatially smaller than     │
# │  activation effects. The 7T brain mask has 1.35M voxels — Bonferroni   │
# │  requires z > 5.13. Surviving voxels are few and isolated; a cluster   │
# │  threshold of 10 erases them. Set to 0 for an initial exploration.     │
# └─────────────────────────────────────────────────────────────────────────┘
ALPHA_FDR         = 0.05 # see table above
ALPHA_BONF        = 0.02  # see table above
CLUSTER_THRESHOLD = 5      # set to 0 for joystick motion method

DISPLAY_THRESHOLD = 1    # Z > 2.3 ≈ p < 0.01 uncorrected (QC plots only)

# --- Event method ---
# motif4limbs : "sequence"  OR  "behavioral"
# joystick    : "events"    OR  "motion"
METHOD = "events"


In [31]:
# ==============================================================================
# SECTION 2 — PATHS
# ==============================================================================

FMRIPREP_ROOT    = Path("/neurospin/motif-stroke/7T_protocol/pilots/derivatives/fmriprep")
BEHAVIORAL_ROOT  = Path("/neurospin/motif-stroke/7T_protocol/pilots/Behavioral_data")
HOME             = Path.home()
BASE_DIR         = HOME / "7T-fMRI-Motor-Stroke"

# Results folder named after the task automatically
RESULTS_DIR = BASE_DIR / f"results_{TASK}_V01"

# --- Behavioral sub-folder name per task ---
_TASK_FOLDER = "4limbs" if TASK == "motif4limbs" else "joystick"
_BEH_DIR     = BEHAVIORAL_ROOT / f"sub-{SUB}" / _TASK_FOLDER

# motif4limbs
SEQ_DIR  = _BEH_DIR / "seq"    # stim_sequence_run-*.csv
LOGS_DIR = _BEH_DIR / "log"    # raw .csv logs

# both tasks — cleaned outputs (events.tsv + motion.tsv for joystick)
PARADIGM_DIR = _BEH_DIR / "output_paradigm_descriptors"

In [ ]:
# ==============================================================================
# SECTION 3 — PER-TASK CONFIGURATION
# Each task/method combination has its own contrasts, ROI coords, and views.
# To switch task or method, only change Section 1. Nothing here needs editing.
# ==============================================================================

# ------------------------------------------------------------------------------
# 3A — MOTIF4LIMBS  (shared by "sequence" and "behavioral" methods)
# Conditions: main_gauche, main_droite, pied_gauche, pied_droit
# ------------------------------------------------------------------------------
MOTIF4LIMBS_CONFIG = {

    "contrasts": {
        "task_gt_baseline":    "0.25*(main_gauche + main_droite + pied_gauche + pied_droit)",
        "hand_vs_foot":        "0.5*(main_gauche + main_droite) - 0.5*(pied_gauche + pied_droit)",
        "foot_vs_hand":        "0.5*(pied_gauche + pied_droit) - 0.5*(main_gauche + main_droite)",
        "right_vs_left":       "0.5*(main_droite + pied_droit) - 0.5*(main_gauche + pied_gauche)",
        "right_vs_left_hand":  "main_droite - main_gauche",
        "right_vs_left_foot":  "pied_droit - pied_gauche",
    },

    "roi_coords": {
        "SMA":            (0,  -5,  65),
        "M1_hand_L":      (-40, -25, 55),
        "M1_hand_R":      (40,  -25, 55),
        "M1_foot_medial": (0,  -30,  70),
        "overview":       (0,  -20,  60),
    },

    "contrast_views": {
        "task_gt_baseline":   ["overview", "SMA"],
        "hand_vs_foot":       ["M1_hand_L", "M1_hand_R", "SMA"],
        "foot_vs_hand":       ["M1_hand_L", "M1_hand_R", "SMA"],
        "right_vs_left":      ["overview", "SMA"],
        "right_vs_left_hand": ["M1_hand_L", "M1_hand_R", "SMA"],
        "right_vs_left_foot": ["M1_foot_medial", "SMA"],
    },
}


# ------------------------------------------------------------------------------
# 3B — JOYSTICK / Events method
# *_reached events DROPPED from the design matrix (see design_matrix_audit.md):
#   - left ↔ left_reached: r=0.92 (interval ~1s → nearly identical HRFs)
#   - Dropping *_reached eliminates collinearity entirely
#   - goal_attained_feedback and active_effort_vs_passive were already removed
#   - No scientifically usable contrast relied on *_reached
# Conditions: left, right, center  (modulation = 1.0)
# Asks: WHERE is the brain active during each push direction?
# ------------------------------------------------------------------------------
JOYSTICK_EVENTS_CONFIG = {

    "contrasts": {
        "task_gt_baseline":   "left + right + center",
        "push_right_vs_left": "right - left",
    },

    "roi_coords": {
        "M1_hand_L": (-38, -22, 56),
        "M1_hand_R": (38,  -22, 56),
        "SMA":       (0,   -8,  62),
        "overview":  (0,   -20, 60),
    },

    "contrast_views": {
        "task_gt_baseline":   ["overview", "SMA"],
        "push_right_vs_left": ["M1_hand_L", "M1_hand_R", "overview"],
    },
}


# ------------------------------------------------------------------------------
# 3C — JOYSTICK / Motion method
# Valid contrasts only (collinear/null contrasts removed — see design_matrix_audit.md):
#   REMOVED: amplitude_active_vs_passive  (symmetric displacement left≈center — null result)
# Conditions: left, right, center  (modulation = peak joystick displacement)
# Parametric GLM  →  asks WHERE does BOLD scale with movement amplitude?
# ------------------------------------------------------------------------------
JOYSTICK_MOTION_CONFIG = {

    "contrasts": {
        "amplitude_modulation":    "left + right + center",
        "amplitude_right_vs_left": "right - left",
    },

    "roi_coords": {
        "M1_hand_L":  (-38, -22, 56),
        "M1_hand_R":  (38,  -22, 56),
        "Cerebellum": (18,  -52, -18),
        "overview":   (0,   -20, 60),
    },

    "contrast_views": {
        "amplitude_modulation":    ["M1_hand_L", "Cerebellum", "overview"],
        "amplitude_right_vs_left": ["M1_hand_L", "M1_hand_R"],
    },
}


# ------------------------------------------------------------------------------
# Config registry — maps (task, method) → config dict.
# get_task_config() does a simple two-level lookup; no special cases needed.
# ------------------------------------------------------------------------------
_CONFIGS = {
    "motif4limbs": {
        "sequence":   MOTIF4LIMBS_CONFIG,
        "behavioral": MOTIF4LIMBS_CONFIG,
    },
    "joystick": {
        "events": JOYSTICK_EVENTS_CONFIG,
        "motion": JOYSTICK_MOTION_CONFIG,
    },
}


In [33]:
def get_task_config():
    """Returns the config dict for the current TASK + METHOD combination."""
    if TASK not in _CONFIGS:
        raise ValueError(f"Unknown TASK '{TASK}'. Choose from: {list(_CONFIGS)}")
    if METHOD not in _CONFIGS[TASK]:
        raise ValueError(f"Unknown METHOD '{METHOD}' for task '{TASK}'. Choose from: {list(_CONFIGS[TASK])}")
    return _CONFIGS[TASK][METHOD]

In [34]:
# ==============================================================================
# SECTION 4 — FILE FINDERS
# ==============================================================================

def get_bold_files(sub_id):
    """
    Finds all BOLD runs for the current TASK and SPACE.
    Works for both motif4limbs and joystick.
    Returns a list of dicts with keys: run, dir, bold, mask, conf.
    """
    func_dir = FMRIPREP_ROOT / f"sub-{sub_id}" / "func"

    if SPACE == "T1w":
        pattern = f"sub-{sub_id}_task-{TASK}_dir-*_run-*_desc-preproc_bold.nii.gz"
    else:
        pattern = f"sub-{sub_id}_task-{TASK}_dir-*_run-*_space-{SPACE}_desc-preproc_bold.nii.gz"

    bolds = sorted(func_dir.glob(pattern))
    if not bolds:
        print(f"  ⚠️  No BOLD files found for sub-{sub_id}, task-{TASK}, space-{SPACE}")

    run_data = []
    for b in bolds:
        run_match = re.search(r"run-(\d+)", b.name)
        dir_match = re.search(r"dir-([a-zA-Z]+)", b.name)   # fix: was [a-z]+, missed AP/PA
        if not (run_match and dir_match):
            continue

        run_num = run_match.group(1)
        direc   = dir_match.group(1)

        run_data.append({
            "run":  run_num,
            "dir":  direc,
            "bold": b,
            "mask": func_dir / b.name.replace("desc-preproc_bold.nii.gz",
                                               "desc-brain_mask.nii.gz"),
            "conf": func_dir / b.name.replace(f"space-{SPACE}_desc-preproc_bold.nii.gz",
                                               "desc-confounds_timeseries.tsv"),
        })
    return run_data


def get_background_image(sub_id):
    """
    Returns the T1 background image path for the chosen SPACE.
    Uses glob to handle optional BIDS entities (e.g. rec-uniden) that vary
    between subjects depending on how the T1w was named during import.
    """
    anat_dir = FMRIPREP_ROOT / f"sub-{sub_id}" / "anat"
    if SPACE == "T1w":
        pattern = f"sub-{sub_id}_*desc-preproc_T1w.nii.gz"
    else:
        pattern = f"sub-{sub_id}_*space-{SPACE}_desc-preproc_T1w.nii.gz"
    matches = sorted(anat_dir.glob(pattern))
    return str(matches[0]) if matches else None


In [ ]:
# ==============================================================================
# SECTION 5 — EVENT LOADERS (one per task / method)
# ==============================================================================

def load_events(run_id, TASK=TASK, METHOD=METHOD):
    """
    Dispatcher: selects the correct event loader based on TASK and METHOD.
    Always returns a Nilearn-ready DataFrame with columns:
        onset (s), duration (s), trial_type, modulation
    """
    run_int = int(run_id)

    if TASK == "motif4limbs":
        if METHOD == "sequence":
            return _events_motif_sequence(run_int)
        elif METHOD == "behavioral":
            log_files = sorted(LOGS_DIR.glob(f"*sub-{int(SUB)}_run-{run_int}*.csv"))
            if not log_files:
                raise FileNotFoundError(f"No log file for sub-{SUB} run-{run_id}")
            return _events_motif_behavioral(log_files[0])
        else:
            raise ValueError(f"Unknown METHOD '{METHOD}' for motif4limbs.")

    elif TASK == "joystick":
        if METHOD == "events":
            matches = sorted(PARADIGM_DIR.glob(f"*run-{run_int}*events.tsv"))
            if not matches:
                raise FileNotFoundError(
                    f"No events TSV for sub-{SUB} run-{run_id} in {PARADIGM_DIR}")
            return _events_joystick(matches[0])
        elif METHOD == "motion":
            return _events_joystick_motion(run_int)
        else:
            raise ValueError(f"Unknown METHOD '{METHOD}' for joystick.")

    else:
        raise ValueError(f"Unknown TASK '{TASK}'.")


# ------------------------------------------------------------------------------
def _events_motif_sequence(run_int):
    """
    Motif4Limbs / sequence method.
    Reconstructs onsets from the stimulus sequence CSV using fixed timing.
    """
    seq_path = SEQ_DIR / f"stim_sequence_run-{run_int}.csv"
    if not seq_path.exists():
        return None
    seq = pd.read_csv(seq_path).sort_values(["block_id", "id"])

    rows = []
    current = 0.0
    current_block = None

    for _, r in seq.iterrows():
        if current_block is None:
            current_block = r["block_id"]
        elif r["block_id"] != current_block:
            current += PAUSE_S
            current_block = r["block_id"]

        current += BLANK_S
        rows.append({
            "onset":      current,
            "duration":   STIM_S,
            "trial_type": r["block_name"],
            "modulation": 1.0,
        })
        current += STIM_S

    return pd.DataFrame(rows) if rows else None


# ------------------------------------------------------------------------------
def _events_motif_behavioral(log_csv_path):
    """
    Motif4Limbs / behavioral method.
    Extracts keypresses from the raw log, anchored to TTL pulse.
    Handles two log formats:
      - Old (sub-01..05): key-code events (key:49, key:50, key:98, key:121)
      - New (sub-06+):    stimulus names logged directly (pied_droit, main_gauche, ...)
    """
    df = pd.read_csv(log_csv_path)
    df['event'] = df['event'].astype(str).apply(
        lambda x: x.split(':')[1] if 'key:' in x else x)

    pulses = df[df['event'].isin(['TTL', '116'])]
    t0 = pulses.iloc[0]['time']

    # Old format: numeric key codes
    response_map = {
        '49':  'pied_droit',
        '50':  'pied_gauche',
        '98':  'main_droite',
        '121': 'main_gauche',
    }
    # New format: stimulus names already in the log
    response_map_names = {
        'pied_droit':  'pied_droit',
        'pied_gauche': 'pied_gauche',
        'main_droite': 'main_droite',
        'main_gauche': 'main_gauche',
    }

    df['event_str'] = df['event'].astype(str).str.strip()
    # Auto-detect: use key codes if present, else fall back to stimulus names
    if not df['event_str'].isin(response_map).any():
        response_map = response_map_names
    df_beh = df[df['event_str'].isin(response_map.keys())].copy()

    df_beh['onset']      = (df_beh['time'] - t0) / 1000.0
    df_beh['duration']   = 0.5
    df_beh['trial_type'] = df_beh['event_str'].map(response_map)
    df_beh['modulation'] = 1.0

    return df_beh[['onset', 'duration', 'trial_type', 'modulation']]


# ------------------------------------------------------------------------------
def _events_joystick(tsv_path):
    """
    Joystick / events method.
    Loads the BIDS events TSV, keeps only push-direction events (left, right, center).
    *_reached events are dropped: intervals of ~1s between push and reach produce
    r=0.92 HRF collinearity, making *_reached regressors indistinguishable from
    the push regressors. No valid contrast required *_reached.
    Modulation = 1.0  →  asks WHERE is the brain active during each push direction.
    """
    df = pd.read_csv(tsv_path, sep='\t')

    # Keep only the three push-direction conditions
    keep = {'left', 'right', 'center'}
    df = df[df['trial_type'].isin(keep)].copy()

    # Convert ms → s if values are in milliseconds
    if df['onset'].max() > 1000:
        df['onset']    = df['onset']    / 1000.0
        df['duration'] = df['duration'] / 1000.0

    df['modulation'] = 1.0
    return df[['onset', 'duration', 'trial_type', 'modulation']]


# ------------------------------------------------------------------------------
def _events_joystick_motion(run_int):
    """
    Joystick / motion method.
    Builds a parametric events table: one row per trial (left / right / center),
    where modulation = peak Euclidean joystick displacement during that trial.

    Design matrix conditions: left, right, center
    Each is weighted by how far the joystick actually moved — so the GLM
    estimates where BOLD scales proportionally with movement amplitude.
    """
    ev_matches  = sorted(PARADIGM_DIR.glob(f"*run-{run_int}*events.tsv"))
    mot_matches = sorted(PARADIGM_DIR.glob(f"*run-{run_int}*motion.tsv"))
    ev  = pd.read_csv(ev_matches[0],  sep='\t')
    mot = pd.read_csv(mot_matches[0], sep='\t')

    # Convert timestamps to seconds; compute Euclidean distance from joystick centre
    ev['onset']   = ev['onset']   / 1000.0
    mot['time_s'] = mot['time']   / 1000.0
    mot['dist']   = np.sqrt(mot['gamepad_posx'].astype(float)**2 +
                            mot['gamepad_posy'].astype(float)**2)

    rows  = []
    ignore = {'MR_trigger', 'fixation', 'rest', 'Start', 'End', 'fix'}
    starts = ev[
        ~ev['trial_type'].astype(str).str.contains('_reached') &
        ~ev['trial_type'].isin(ignore)
    ].index

    for idx in starts:
        onset_s = ev.loc[idx, 'onset']
        t_type  = ev.loc[idx, 'trial_type']

        # Duration = time until the matching *_reached event
        end_label  = f"{t_type}_reached"
        future     = ev.iloc[idx + 1:]
        reached    = future[future['trial_type'] == end_label]
        duration_s = max(0.1, reached.iloc[0]['onset'] - onset_s) \
                     if not reached.empty else 1.0

        # Modulation = peak joystick displacement (×100 for numerical scaling)
        mask     = (mot['time_s'] >= onset_s) & (mot['time_s'] <= onset_s + duration_s)
        peak_val = mot[mask]['dist'].max() * 100 if not mot[mask].empty else 1.0

        rows.append({
            'onset':      onset_s,
            'duration':   duration_s,
            'trial_type': t_type,
            'modulation': peak_val,
        })

    return pd.DataFrame(rows)


In [36]:
# ==============================================================================
# SECTION 6 — GLM FUNCTIONS
# ==============================================================================
 
def build_design_matrix(bold_img, events, conf_path):
    """Builds the first-level design matrix with motion confounds."""
    n_scans     = bold_img.shape[-1]
    tr          = bold_img.header.get_zooms()[-1]
    frame_times = np.arange(n_scans) * tr
 
    conf = pd.read_csv(conf_path, sep="\t")
    motion_cols = [c for c in
                   ["trans_x","trans_y","trans_z","rot_x","rot_y","rot_z"]
                   if c in conf.columns]
    conf_sel = conf[motion_cols].fillna(0.0)
 
    return make_first_level_design_matrix(
        frame_times=frame_times,
        events=events,
        hrf_model='glover',
        drift_model='cosine',
        high_pass=0.01,
        add_regs=conf_sel.values,
        add_reg_names=list(conf_sel.columns),
    )

def fit_run(bold_path, mask_path, dm, run_dir, c_name, c_expr,
            smoothing_fwhm=SMOOTHING_FWHM):
    """
    Fits the GLM for one run and saves per-run NIfTIs.
    Returns the stats dict (z_score, effect_size, effect_variance).
    """
    bold_img = nib.load(str(bold_path))
    tr       = bold_img.header.get_zooms()[-1]
 
    model = FirstLevelModel(
        t_r=tr,
        mask_img=str(mask_path),
        hrf_model='glover',
        drift_model='cosine',
        high_pass=0.01,
        noise_model='ar1',
        smoothing_fwhm=smoothing_fwhm,
        standardize=False,
        minimize_memory=False,
    ).fit(bold_img, design_matrices=dm)
 
    stats = model.compute_contrast(c_expr, output_type='all')
 
    stats['z_score'].to_filename(       run_dir / f"{c_name}_zmap.nii.gz")
    stats['effect_size'].to_filename(   run_dir / f"{c_name}_beta.nii.gz")
    stats['effect_variance'].to_filename(run_dir / f"{c_name}_variance.nii.gz")
 
    return stats

def fuse_runs(run_stats_list, mask_path, out_dir, c_name,
              alpha_fdr=ALPHA_FDR, alpha_bonf=ALPHA_BONF,
              cluster_thresh=CLUSTER_THRESHOLD):
    """
    Fixed-effects fusion across runs, applies FDR and Bonferroni corrections.
    Saves all NIfTIs and returns a results dict.
    """
    out_dir.mkdir(parents=True, exist_ok=True)
 
    eff_imgs = [s['effect_size']    for s in run_stats_list]
    var_imgs = [s['effect_variance'] for s in run_stats_list]
 
    contrast_map, var_map, t_map, z_map = compute_fixed_effects(
        eff_imgs, var_imgs, mask_path)
 
    z_fdr,  fdr_thresh  = threshold_stats_img(
        z_map, alpha=alpha_fdr,  height_control='fdr',
        cluster_threshold=cluster_thresh)
    z_bonf, bonf_thresh = threshold_stats_img(
        z_map, alpha=alpha_bonf, height_control='bonferroni',
        cluster_threshold=cluster_thresh)
 
    contrast_map.to_filename(out_dir / f"{c_name}_TOTAL_beta.nii.gz")
    var_map.to_filename(     out_dir / f"{c_name}_TOTAL_variance.nii.gz")
    t_map.to_filename(       out_dir / f"{c_name}_TOTAL_tstat.nii.gz")
    z_map.to_filename(       out_dir / f"{c_name}_TOTAL_zmap_uncorrected.nii.gz")
    z_fdr.to_filename(       out_dir / f"{c_name}_zmap_FDR_q{alpha_fdr}.nii.gz")
    z_bonf.to_filename(      out_dir / f"{c_name}_zmap_BONF_a{alpha_bonf}.nii.gz")
 
    return {
        'beta':            contrast_map,
        'variance':        var_map,
        't_stat':          t_map,
        'z_uncorrected':   z_map,
        'z_fdr':           z_fdr,
        'fdr_threshold':   fdr_thresh,
        'z_bonf':          z_bonf,
        'bonf_threshold':  bonf_thresh,
    }

In [37]:
# ==============================================================================
# SECTION 7 — VISUALIZATION
# ==============================================================================
 
def save_brain_viz(nifti_img, output_png, title, sub_id,
                   coords=(0, -30, 60), threshold=DISPLAY_THRESHOLD):
    """Saves an ortho stat map PNG with the subject's own T1 as background."""
    bg = get_background_image(sub_id)
    display = plot_stat_map(
        nifti_img,
        bg_img=bg,
        threshold=threshold,
        cut_coords=coords,
        title=title,
        colorbar=True,
        display_mode='ortho',
    )
    plt.savefig(output_png)
    plt.close()
 
 
def find_peak_coords(z_map, mask_img=None):
    """
    Returns MNI coordinates of the peak activation voxel.
    Returns None if no suprathreshold cluster is found.
    """
    try:
        return find_xyz_cut_coords(img=z_map, mask_img=mask_img)
    except Exception:
        return None

In [38]:
# ==============================================================================
# SECTION 8 — MASTER PIPELINE DRIVER
# ==============================================================================

def run_pipeline():
    cfg = get_task_config()
    CONTRASTS      = cfg["contrasts"]
    ROI_COORDS     = cfg["roi_coords"]
    CONTRAST_VIEWS = cfg["contrast_views"]

    # --- Fetch and filter runs ---
    data_runs = get_bold_files(SUB)
    data_runs = [r for r in data_runs if r['run'] not in EXCLUDE_RUNS]

    if not data_runs:
        print(f"❌ No usable runs found for sub-{SUB}, task-{TASK}. Aborting.")
        return

    # Filename prefix and results root include subject + task + method
    # → results_joystick_V01/sub-06/events/execution_vs_return/...
    prefix   = f"sub-{SUB}_{TASK}_{METHOD}"
    base_dir = RESULTS_DIR / f"sub-{SUB}" / METHOD

    print(f"\n{'='*65}")
    print(f"  TASK     : {TASK.upper()}")
    print(f"  SUBJECT  : sub-{SUB}")
    print(f"  SPACE    : {SPACE}")
    print(f"  METHOD   : {METHOD}")
    print(f"  RUNS     : {[r['run'] for r in data_runs]}")
    print(f"  EXCLUDED : {EXCLUDE_RUNS}")
    print(f"  CONTRASTS: {list(CONTRASTS.keys())}")
    print(f"  OUT DIR  : {base_dir}")
    print(f"{'='*65}\n")

    for c_name, c_expr in CONTRASTS.items():
        print(f"\n── CONTRAST: {c_name} ──")

        sub_dir = base_dir / c_name
        sub_dir.mkdir(parents=True, exist_ok=True)

        views          = CONTRAST_VIEWS.get(c_name, ["overview"])
        run_stats_list = []

        # ── STEP A: fit each run individually ─────────────────────────────
        for run_data in data_runs:
            run_id  = run_data['run']
            run_dir = sub_dir / f"run-{run_id}_dir-{run_data['dir']}"
            run_dir.mkdir(exist_ok=True)

            print(f"  [Run {run_id}] Loading events...")
            ev = load_events(run_id, TASK=TASK, METHOD=METHOD)

            print(f"  [Run {run_id}] Building design matrix...")
            dm = build_design_matrix(nib.load(str(run_data['bold'])),
                                     ev, run_data['conf'])

            print(f"  [Run {run_id}] Fitting GLM...")
            stats = fit_run(run_data['bold'], run_data['mask'],
                            dm, run_dir, c_name, c_expr)
            run_stats_list.append(stats)

            # QC brain maps per run
            for v_name in views:
                save_brain_viz(
                    stats['z_score'],
                    run_dir / f"{prefix}_{c_name}_run-{run_id}_{v_name}.png",
                    title=f"Run {run_id} | {v_name} | {c_name}",
                    sub_id=SUB,
                    coords=ROI_COORDS.get(v_name, (0, 0, 0)),
                    threshold=3.1,
                )

        # ── STEP B: fixed-effects fusion ──────────────────────────────────
        print(f"  [Fusion] Combining {len(run_stats_list)} runs...")
        total_dir = sub_dir / "combined_total"
        results = fuse_runs(
            run_stats_list, data_runs[0]['mask'],
            total_dir, c_name,
        )

        # ── STEP C: peak discovery ────────────────────────────────────────
        peak = find_peak_coords(results['z_fdr'], data_runs[0]['mask'])
        if peak is not None:
            print(f"  ⭐ Peak voxel (FDR): {np.round(peak, 1)}")

        # ── STEP D: final scientific maps ─────────────────────────────────
        final_views = views.copy()
        if peak is not None:
            final_views.append("AUTO_PEAK")

        for v_name in final_views:
            if v_name == "AUTO_PEAK":
                coords  = peak
                v_label = "peak_" + "_".join(str(int(x)) for x in np.round(peak, 0))
            else:
                coords  = ROI_COORDS.get(v_name, (0, 0, 0))
                v_label = v_name

            save_brain_viz(
                results['z_fdr'],
                total_dir / f"{prefix}_{c_name}_FDR_q{ALPHA_FDR}_{v_label}.png",
                title=(f"FDR q<{ALPHA_FDR} | {v_label} | "
                       f"Z>{results['fdr_threshold']:.2f} | {c_name}"),
                sub_id=SUB, coords=coords, threshold=1e-6,
            )
            save_brain_viz(
                results['z_bonf'],
                total_dir / f"{prefix}_{c_name}_BONF_a{ALPHA_BONF}_{v_label}.png",
                title=(f"Bonf a<{ALPHA_BONF} | {v_label} | "
                       f"Z>{results['bonf_threshold']:.2f} | {c_name}"),
                sub_id=SUB, coords=coords, threshold=1e-6,
            )

        del results  # free memory before next contrast

    print(f"\n✅ PIPELINE COMPLETE — sub-{SUB} / task-{TASK} / method-{METHOD}\n")

In [39]:
# ==============================================================================
# ENTRY POINT
# ==============================================================================
run_pipeline()



  TASK     : MOTIF4LIMBS
  SUBJECT  : sub-07
  SPACE    : MNI152NLin2009cAsym
  METHOD   : sequence
  RUNS     : ['1', '2']
  EXCLUDED : []
  CONTRASTS: ['task_gt_baseline', 'hand_vs_foot', 'foot_vs_hand', 'right_vs_left', 'right_vs_left_hand', 'right_vs_left_foot']
  OUT DIR  : /home/sb283337/7T-fMRI-Motor-Stroke/results_motif4limbs_V01/sub-07/sequence


── CONTRAST: task_gt_baseline ──
  [Run 1] Loading events...
  [Run 1] Building design matrix...
[make_first_level_design_matrix] A 'modulation' column was found in the given events data and is used.
  [Run 1] Fitting GLM...


/tmp/ipykernel_507676/808318483.py:46: UserWarning: If design matrices are supplied, [t_r] will be ignored.
  ).fit(bold_img, design_matrices=dm)


  [Run 2] Loading events...
  [Run 2] Building design matrix...
[make_first_level_design_matrix] A 'modulation' column was found in the given events data and is used.
  [Run 2] Fitting GLM...


/tmp/ipykernel_507676/808318483.py:46: UserWarning: If design matrices are supplied, [t_r] will be ignored.
  ).fit(bold_img, design_matrices=dm)


  [Fusion] Combining 2 runs...
  ⭐ Peak voxel (FDR): [ -8.1 -80.8  36.4]

── CONTRAST: hand_vs_foot ──
  [Run 1] Loading events...
  [Run 1] Building design matrix...
[make_first_level_design_matrix] A 'modulation' column was found in the given events data and is used.
  [Run 1] Fitting GLM...


...

  [Fusion] Combining 2 runs...
  ⭐ Peak voxel (FDR): [ 38.5 -19.5  51.4]

── CONTRAST: right_vs_left_foot ──
  [Run 1] Loading events...
  [Run 1] Building design matrix...
[make_first_level_design_matrix] A 'modulation' column was found in the given events data and is used.
  [Run 1] Fitting GLM...


/tmp/ipykernel_507676/808318483.py:46: UserWarning: If design matrices are supplied, [t_r] will be ignored.
  ).fit(bold_img, design_matrices=dm)


  [Run 2] Loading events...
  [Run 2] Building design matrix...
[make_first_level_design_matrix] A 'modulation' column was found in the given events data and is used.
  [Run 2] Fitting GLM...


/tmp/ipykernel_507676/808318483.py:46: UserWarning: If design matrices are supplied, [t_r] will be ignored.
  ).fit(bold_img, design_matrices=dm)


  [Fusion] Combining 2 runs...
  ⭐ Peak voxel (FDR): [ -4.6 -26.9  71.3]

✅ PIPELINE COMPLETE — sub-07 / task-motif4limbs / method-sequence

